# Reproduksi BSARec (ML-1M)
Urutan: **A** setup → **B** baseline (hasil sudah ada, jangan dijalankan ulang)

# A. Setup


In [ ]:
%cd /content
![ -d BSARec ] || git clone https://github.com/yehjin-shin/BSARec.git
%cd /content/BSARec/src
!pip install -q scipy tqdm wandb

/content
Cloning into 'BSARec'...
remote: Enumerating objects: 592, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 592 (delta 28), reused 20 (delta 20), pack-reused 549 (from 1)
Receiving objects: 100% (592/592), 859.10 MiB | 19.04 MiB/s, done.
Resolving deltas: 100% (136/136), done.
Updating files: 100% (384/384), done.
/content/BSARec/src


# B. Baseline (`main.py` asli)
### B1. Running BSARec
Sel di bagian ini sudah dijalankan dan outputnya disimpan sebagai bukti hasil reproduksi. **Jangan dijalankan ulang**


In [ ]:
!python main.py \
  --data_name ML-1M \
  --model_type BSARec \
  --num_workers 2 \
  --batch_size 256 \
  --lr 0.001 \
  --hidden_size 64 \
  --num_hidden_layers 2 \
  --num_attention_heads 2

# python main.py --data_name ML-1M --model_type BSARec --c 3 --alpha 0.9 --num_workers 0

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
2026-10-29 17:04:11,265 - Namespace(data_dir='./data/', output_dir='output/', data_name='ML-1M', do_eval=False, load_model=None, train_name='Oct-29-2026_17-04-10', num_items=10, num_users=6041, lr=0.001, batch_size=256, epochs=200, no_cuda=False, log_freq=1, patience=10, num_workers=4, seed=42, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, gpu_id='0', variance=5, model_type='BSARec', max_seq_length=50, hidden_size=64, num_hidden_layers=2, hidden_act='gelu', num_attention_heads=2, attention_probs_dropout_prob=0

### B2. Running SASRec
Sel di bagian ini sudah dijalankan dan outputnya disimpan sebagai bukti hasil reproduksi. **Jangan dijalankan ulang**

In [ ]:
!python main.py \
  --data_name ML-1M \
  --model_type SASRec \
  --num_workers 2 \
  --batch_size 256 \
  --lr 0.001 \
  --hidden_size 64 \
  --num_hidden_layers 2 \
  --num_attention_heads 2

2026-10-29 15:54:02,524 - Namespace(data_dir='./data/', output_dir='output/', data_name='ML-1M', do_eval=False, load_model=None, train_name='Oct-29-2026_15-54-00', num_items=10, num_users=6041, lr=0.001, batch_size=256, epochs=200, no_cuda=False, log_freq=1, patience=10, num_workers=2, seed=42, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, gpu_id='0', variance=5, model_type='SASRec', max_seq_length=50, hidden_size=64, num_hidden_layers=2, hidden_act='gelu', num_attention_heads=2, attention_probs_dropout_prob=0.5, hidden_dropout_prob=0.5, initializer_range=0.02, cuda_condition=True, data_file='./data/ML-1M.txt', item_size=3417, checkpoint_path='output/Oct-29-2026_15-54-00.pt', same_target_path='./data/ML-1M_same_target.npy')
2026-10-29 15:54:02,550 - SASRecModel(
  (item_embeddings): Embedding(3417, 64, padding_idx=0)
  (position_embeddings): Embedding(50, 64)
  (LayerNorm): LayerNorm()
  (dropout): Dropout(p=0.5, inplace=False)
  (item_encoder): TransformerEncoder(
    (blocks): 

# C. Perbandingan hasil BSARec dan SASRec di device dengan di paper Shin et al. (2024)


In [ ]:
# Sel Colab: bandingkan hasil reproduksi ML-1M dengan Table 2 paper
# Semua angka diketik manual, tidak butuh file. Ganti SELURUH isi sel perbandingan lama dengan ini.
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

KEYS = ["HR@5", "NDCG@5", "HR@10", "NDCG@10", "HR@20", "NDCG@20"]

# 'HR@5': '0.1720', 'NDCG@5': '0.1151', 'HR@10': '0.2609', 'NDCG@10': '0.1438', 'HR@20': '0.3652', 'NDCG@20': '0.1701'}
paper = {  # Table 2 paper, ML-1M
    "BSARec": [0.1944, 0.1306, 0.2757, 0.1568, 0.3884, 0.1851],
    "SASRec": [0.1374, 0.0873, 0.2137, 0.1116, 0.3245, 0.1395]
}
hasilku = {  # hasil test akhir per seed (urutan: HR@5, NDCG@5, HR@10, NDCG@10, HR@20, NDCG@20)
    "BSARec": {"s42": [0.1720, 0.1251, 0.2609, 0.1438, 0.3652, 0.1701]},
    "SASRec": {"s42": [0.1308, 0.0852, 0.2084, 0.1100, 0.3199, 0.1380]}
}

hasil = {}
for model in paper:
    df = pd.DataFrame(hasilku[model], index=KEYS)
    seeds = list(df.columns)
    df.insert(0, "Paper", paper[model])
    df["Mean"] = df[seeds].mean(axis=1)
    df["Std"] = df[seeds].std(axis=1)
    for s in seeds:
        df[f"Gap {s} (%)"] = (df[s] - df["Paper"]) / df["Paper"] * 100
    df["Gap Mean (%)"] = (df["Mean"] - df["Paper"]) / df["Paper"] * 100
    hasil[model] = df
    print(f"\n=== {model} (ML-1M), {len(seeds)} seed: {', '.join(seeds)} ===")
    print(df.round(4).to_string())

# Perbandingan keseluruhan: seberapa besar BSARec mengungguli SASRec (paper vs reproduksi)
b, s = hasil["BSARec"], hasil["SASRec"]
ringkas = pd.DataFrame({
    "BSARec paper": b["Paper"], "BSARec mean": b["Mean"],
    "SASRec paper": s["Paper"], "SASRec mean": s["Mean"],
    "Naik vs SASRec, paper (%)": (b["Paper"] / s["Paper"] - 1) * 100,
    "Naik vs SASRec, aku (%)": (b["Mean"] / s["Mean"] - 1) * 100,
})
print("\n=== Ringkasan: BSARec vs SASRec ===")
print(ringkas.round(4).to_string())


=== BSARec (ML-1M), 1 seed: s42 ===
          Paper     s42    Mean  Std  Gap s42 (%)  Gap Mean (%)
HR@5     0.1944  0.1720  0.1720  NaN     -11.5226      -11.5226
NDCG@5   0.1306  0.1251  0.1251  NaN      -4.2113       -4.2113
HR@10    0.2757  0.2609  0.2609  NaN      -5.3682       -5.3682
NDCG@10  0.1568  0.1438  0.1438  NaN      -8.2908       -8.2908
HR@20    0.3884  0.3652  0.3652  NaN      -5.9732       -5.9732
NDCG@20  0.1851  0.1701  0.1701  NaN      -8.1037       -8.1037

=== SASRec (ML-1M), 1 seed: s42 ===
          Paper     s42    Mean  Std  Gap s42 (%)  Gap Mean (%)
HR@5     0.1374  0.1308  0.1308  NaN      -4.8035       -4.8035
NDCG@5   0.0873  0.0852  0.0852  NaN      -2.4055       -2.4055
HR@10    0.2137  0.2084  0.2084  NaN      -2.4801       -2.4801
NDCG@10  0.1116  0.1100  0.1100  NaN      -1.4337       -1.4337
HR@20    0.3245  0.3199  0.3199  NaN      -1.4176       -1.4176
NDCG@20  0.1395  0.1380  0.1380  NaN      -1.0753       -1.0753

=== Ringkasan: BSARec vs SASR

### C1. Penjelasan hasil/perbandingan

1. Evaluasi BSARec (ML-1M, Seed: s42)

Bagian pertama membandingkan metrik performa BSARec hasil run (s42) dengan angka yang dipublikasikan di paper (Paper):

* HR@5: Paper 0.1944 vs Reproduksi 0.1720 (Gap: -11.52%)
* NDCG@5: Paper 0.1306 vs Reproduksi 0.1251 (Gap: -4.21%)
* HR@10: Paper 0.2757 vs Reproduksi 0.2609 (Gap: -5.37%)
* NDCG@10: Paper 0.1568 vs Reproduksi 0.1438 (Gap: -8.29%)
* HR@20: Paper 0.3884 vs Reproduksi 0.3652 (Gap: -5.97%)
* NDCG@20: Paper 0.1851 vs Reproduksi 0.1701 (Gap: -8.10%)

Analisis: Performa BSARec pada eksperimen sedikit di bawah klaim paper dengan penurunan (gap) rata-rata berkisar antara 4.2% hingga 11.5%.

2. Evaluasi SASRec (ML-1M, Seed: s42)

Bagian kedua memperlihatkan perbandingan model baseline SASRec:
* HR@5: Paper 0.1374 vs Reproduksi 0.1308 (Gap: -4.80%)   
* NDCG@5: Paper 0.0873 vs Reproduksi 0.0852 (Gap: -2.41%)
* HR@10: Paper 0.2137 vs Reproduksi 0.2084 (Gap: -2.48%)   
* NDCG@10: Paper 0.1116 vs Reproduksi 0.1100 (Gap: -1.43%)   
* HR@20: Paper 0.3245 vs Reproduksi 0.3199 (Gap: -1.42%)   
* NDCG@20: Paper 0.1395 vs Reproduksi 0.1380 (Gap: -1.08%)   

Analisis: Hasil reproduksi untuk SASRec sangat presisi dan hampir identik dengan angka paper, dengan rentang gap 1.08%–4.80%.